In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

y = np.log1p(train['SalePrice'])
X_raw = train.drop(['Id', 'SalePrice'], axis=1)
X_test_raw = test.drop(['Id'], axis=1)

print(f"{X_raw.shape} and {X_test_raw.shape}")

(1460, 79) and (1459, 79)


In [3]:
combined = pd.concat([X_raw, X_test_raw], axis=0)
for col in combined.columns:
    if combined[col].dtype in [np.number]: combined[col].fillna(combined[col].median())
    else : combined[col].fillna('None')

combined_encoded = pd.get_dummies(combined)
X_all = combined_encoded.iloc[:len(train)]
X_test_all = combined_encoded.iloc[len(train):]

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y, test_size = 0.2, random_state=42
)

/tmp/ipykernel_16/1301864334.py:3: DeprecationWarning: Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.
  if combined[col].dtype in [np.number]: combined[col].fillna(combined[col].median())


In [4]:
rf_model = RandomForestRegressor(n_estimators=150, max_depth=20, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_train_pred = rf_model.predict(X_train)
y_val_pred = rf_model.predict(X_val)

train_rmse = root_mean_squared_error(y_train, y_train_pred)
val_rmse = root_mean_squared_error(y_val, y_val_pred)

print(f"train rmse= {train_rmse:.4} and val rmse = {val_rmse:.4}")

train rmse= 0.07572 and val rmse = 0.1462


In [5]:
from xgboost import XGBRegressor
model = XGBRegressor(
    n_estimators = 500,
    learning_rate = 0.05,
    max_depth = 4,
    subsample = 0.7,
    colsample_bytree=0.7,
    random_state = 42,
    n_jobs = -1
)
model.fit(X_train, y_train)

xgb_train_rmse = root_mean_squared_error(y_train, model.predict(X_train))
xgb_val_rmse = root_mean_squared_error(y_val, model.predict(X_val))

print(f"XGBoost Training RMSE: {xgb_train_rmse:.4f}")
print(f"XGBoost Validation RMSE: {xgb_val_rmse:.4f}")

XGBoost Training RMSE: 0.0330
XGBoost Validation RMSE: 0.1283


In [6]:
y_test_pred = model.predict(X_test_all)
y_test_pred_dollars = np.expm1(y_test_pred)
submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': y_test_pred_dollars
})

submission.to_csv('submission.csv', index=False)